# Dynamic-A Episodic Validation Run

This notebook wraps `examples/dynamic_episodic_validation.py`. It keeps the script as the canonical result generator while exposing the run hyperparameters in editable cells.

By default the command is only printed. Set `RUN_VALIDATION = True` when you are ready to launch it.

<!-- reviewer-resume-contract -->
## Execution and resume contract

This notebook is aligned with the reviewer-revision implementation. Expensive work is checkpointed and safe to restart with the same configuration. Do not change methods, seeds, thresholds, or output paths while resuming. Saved outputs remain provisional until the compute-machine run and verification gates complete.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "calcium_transient_rising_flank").is_dir():
            return candidate
        nested = candidate / "calcium-transient-rising-flank"
        if (nested / "src" / "calcium_transient_rising_flank").is_dir():
            return nested
    raise RuntimeError("Run from the repository, package root, or notebooks directory.")


PROJECT_ROOT = find_project_root()
PYTHON = sys.executable
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from calcium_transient_rising_flank.checkpointing import format_progress
print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {PYTHON}")

In [ ]:
def command_text(command: list[str]) -> str:
    return " ".join(shlex.quote(str(part)) for part in command)


def run_or_print(command: list[str], *, execute: bool) -> None:
    print(f'[notebook] command: {command_text(command)}', flush=True)
    if not execute:
        print("Dry run only. Set the run toggle to True to execute.")
        return
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_ROOT / "src")
    env["PYTHONUNBUFFERED"] = "1"
    env.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")
    env.setdefault("XDG_CACHE_HOME", "/tmp/font-cache")
    print(format_progress(0, 1, label='Notebook stage') + ' | running', flush=True)
    subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)
    print(format_progress(1, 1, label='Notebook stage') + ' | complete', flush=True)

In [ ]:
RUN_VALIDATION = False

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "validation_results" / "dynamic_episodic_locked"
METHODS = "cgc,cgc-star"
EVENT_MODES = "compressed,physical"
N_SEEDS = 20
N_STEPS = 1500
N_SURROGATES = 1000
MAX_LAG = 1
TAU = None  # set e.g. 2 to test only lag 2
N_PASTS = None  # set e.g. 5 for five conditioning-history samples
SCORE_THRESHOLD = 0.1

DYNAMIC_EPISODES = 3
MIN_RISE_LENGTH = 20
MAX_RISE_LENGTH = 28
RISE_WAVEFORM_LENGTH = 20
TOPOLOGY_MODE = "sequence"  # "sequence" or "generated"
FALL_STATE_MODE = "stochastic_independent"
FALL_INITIAL_SCALE = 1.0
FALL_INITIAL_CEILING_FRACTION = 1.0
EDGE_DROPOUT_PROBABILITY = 0.20
EDGE_ADDITION_PROBABILITY = 0.02
SOURCE_DROPOUT_PROBABILITY = 0.15
SOURCE_RECRUITMENT_PROBABILITY = 0.25
SOURCE_RECRUITMENT_EDGE_PROBABILITY = 0.25
USE_FDR = True
MIN_RISE_RUN_SAMPLES = 1
RISE_CANDIDATE_FILTER = False
RISE_MATCH_MIN_LAG = TAU if TAU is not None else 1
RISE_MATCH_MAX_LAG = TAU if TAU is not None else None
RISE_MATCH_MIN_OVERLAP_SAMPLES = None
RISE_MATCH_MIN_OVERLAP_FRACTION = 0.5
RISE_RUN_CONTEXT_SAMPLES = None  # None uses N_PASTS, or MAX_LAG when N_PASTS is unset


In [ ]:
command = [
    PYTHON,
    "examples/dynamic_episodic_validation.py",
    "--resume",
    "--output-dir",
    str(OUTPUT_DIR),
    "--n-seeds",
    str(N_SEEDS),
    "--n-steps",
    str(N_STEPS),
    "--n-surrogates",
    str(N_SURROGATES),
    "--max-lag",
    str(MAX_LAG),
    "--score-threshold",
    str(SCORE_THRESHOLD),
    "--methods",
    METHODS,
    "--event-modes",
    EVENT_MODES,
    "--dynamic-episodes",
    str(DYNAMIC_EPISODES),
    "--min-rise-length",
    str(MIN_RISE_LENGTH),
    "--max-rise-length",
    str(MAX_RISE_LENGTH),
    "--rise-waveform-length",
    str(RISE_WAVEFORM_LENGTH),
    "--topology-mode",
    TOPOLOGY_MODE,
    "--fall-state-mode",
    FALL_STATE_MODE,
    "--fall-initial-scale",
    str(FALL_INITIAL_SCALE),
    "--fall-initial-ceiling-fraction",
    str(FALL_INITIAL_CEILING_FRACTION),
    "--edge-dropout-probability",
    str(EDGE_DROPOUT_PROBABILITY),
    "--edge-addition-probability",
    str(EDGE_ADDITION_PROBABILITY),
    "--source-dropout-probability",
    str(SOURCE_DROPOUT_PROBABILITY),
    "--source-recruitment-probability",
    str(SOURCE_RECRUITMENT_PROBABILITY),
    "--source-recruitment-edge-probability",
    str(SOURCE_RECRUITMENT_EDGE_PROBABILITY),
    "--min-rise-run-samples",
    str(MIN_RISE_RUN_SAMPLES),
    "--rise-match-min-lag",
    str(RISE_MATCH_MIN_LAG),
    "--rise-match-min-overlap-fraction",
    str(RISE_MATCH_MIN_OVERLAP_FRACTION),
]
if RISE_CANDIDATE_FILTER:
    command.append("--rise-candidate-filter")
if TAU is not None:
    command.extend(["--tau", str(TAU)])
if N_PASTS is not None:
    command.extend(["--n-pasts", str(N_PASTS)])
if RISE_MATCH_MAX_LAG is not None:
    command.extend(["--rise-match-max-lag", str(RISE_MATCH_MAX_LAG)])
if RISE_MATCH_MIN_OVERLAP_SAMPLES is not None:
    command.extend(["--rise-match-min-overlap-samples", str(RISE_MATCH_MIN_OVERLAP_SAMPLES)])
if RISE_RUN_CONTEXT_SAMPLES is not None:
    command.extend(["--rise-run-context-samples", str(RISE_RUN_CONTEXT_SAMPLES)])
if not USE_FDR:
    command.append("--no-fdr")

run_or_print(command, execute=RUN_VALIDATION)


In [ ]:
summary_path = OUTPUT_DIR / "summary.json"
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary.get("config", {}), indent=2))
    print(f"rows: {summary.get('rows')}")
else:
    print(f"No summary yet at {summary_path}")